# 🐾 VetCerca IA - Análisis Exploratorio de Datos (EDA)
**Asignatura:** Prácticas Profesionalizantes I: Aproximación al Campo Laboral  
**Institución:** IFTS N° 12 - Técnico Superior en Ciencia de Datos e Inteligencia Artificial  
**Integrantes:** Caponi Laura, Cortes Alexis, Di Nucci Lara, Gazzola Agostina

---

## Objetivos del Notebook:
1. **Auditoría de Calidad:** Revisión estructural, nulos y tipos de datos en `veterinarias_barrios.csv` y `sintomas_triaje.csv`.
2. **Análisis Territorial:** Distribución de guardias 24hs vs. consultorios diurnos por Comuna y Barrio en CABA.
3. **Análisis Clínico de Triaje:** Correlación entre horas de evolución, estado de conciencia y niveles de severidad.
4. **Frecuencias Cruzadas:** Síntomas principales y urgencia clasificada por especie (Canino / Felino).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

### 1. Carga y Auditoría de Datos

In [ ]:
data_dir = "../data"
path_vets = os.path.join(data_dir, "veterinarias_barrios.csv")
path_triaje = os.path.join(data_dir, "sintomas_triaje.csv")

df_vets = pd.read_csv(path_vets)
df_triaje = pd.read_csv(path_triaje)

print(f"Dataset Veterinarias: {df_vets.shape[0]} filas, {df_vets.shape[1]} columnas.")
print(f"Dataset Triaje: {df_triaje.shape[0]} filas, {df_triaje.shape[1]} columnas.")

In [ ]:
print("=== NULOS EN VETERINARIAS ===")
print(df_vets.isnull().sum())
print("\n=== NULOS EN TRIAJE ===")
print(df_triaje.isnull().sum())

### 2. Cobertura de Guardias 24hs vs. Diurnas por Comuna en CABA

In [ ]:
plt.figure(figsize=(12, 6))
conteo_comuna = df_vets.groupby(["comuna", "guardia_24hs"]).size().unstack(fill_value=0)
conteo_comuna.columns = ["Horario Diurno", "Guardia 24hs"]

conteo_comuna.plot(kind="bar", stacked=True, color=["#4A90E2", "#E74C3C"], figsize=(12, 6))
plt.title("Oferta de Atención Veterinaria por Comuna en CABA (Diurna vs. 24hs)", fontsize=14, fontweight="bold")
plt.xlabel("Comuna Administrativa")
plt.ylabel("Cantidad de Establecimientos")
plt.xticks(rotation=0)
plt.legend(title="Tipo de Servicio")
plt.tight_layout()
plt.show()

### 3. Distribución Geográfica de la Oferta Médica en CABA

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_vets,
    x="longitud",
    y="latitud",
    hue="guardia_24hs",
    style="nivel_complejidad",
    palette={0: "#3498DB", 1: "#E74C3C"},
    s=90,
    alpha=0.9
)
plt.title("Dispersión Geográfica de Clínicas Veterinarias en CABA", fontsize=14, fontweight="bold")
plt.xlabel("Longitud GPS")
plt.ylabel("Latitud GPS")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4. Análisis Clínico de Triaje: Horas de Evolución y Estado de Conciencia vs. Urgencia

In [ ]:
plt.figure(figsize=(10, 5))
orden_urgencia = ["Baja", "Moderada", "Alta", "Emergencia Crítica"]
sns.boxplot(
    data=df_triaje,
    x="nivel_urgencia",
    y="tiempo_evolucion_hs",
    order=orden_urgencia,
    palette="YlOrRd"
)
plt.title("Tiempo de Evolución (Horas) según Nivel de Urgencia", fontsize=14, fontweight="bold")
plt.xlabel("Nivel de Triaje")
plt.ylabel("Horas de Evolución")
plt.tight_layout()
plt.show()

### 5. Matriz de Frecuencia Cruzada: Estado de Conciencia vs. Severidad

In [ ]:
contingencia_conciencia = pd.crosstab(
    df_triaje["estado_conciencia"],
    df_triaje["nivel_urgencia"],
    normalize="index"
)[orden_urgencia]

plt.figure(figsize=(9, 5))
sns.heatmap(contingencia_conciencia, annot=True, fmt=".2%", cmap="Blues", cbar=True)
plt.title("Relación Porcentual: Estado de Conciencia vs. Severidad de Triaje", fontsize=13, fontweight="bold")
plt.ylabel("Estado Neurológico")
plt.xlabel("Nivel de Urgencia Clasificado")
plt.tight_layout()
plt.show()

### 6. Conclusiones del Análisis Exploratorio
1. **Disparidad Territorial:** Las comunas del norte y centro (Palermo, Belgrano, Recoleta) concentran una mayor proporción de guardias 24hs, justificando plenamente el protocolo de contingencia zonal hacia barrios periféricos.
2. **Tiempo y Conciencia:** Los pacientes en estado 'Estuporoso/Comatoso' correlacionan casi unívocamente con 'Emergencia Crítica', lo que confirma la validez predictiva de estas variables en el modelo Random Forest.
3. **Hemorragia:** La presencia de hemorragia eleva inmediatamente el caso al rango de 'Alta' o 'Emergencia Crítica'.